# Phase 5 — Explainable Fuzzy Classifier

**Objective:** Classify seven facial expressions from KDEF using 3–10 interpretable geometric features and a fuzzy inference system.

**Experimental protocol:**
- Original KDEF photographs, frontal pose only.
- Subject-independent 80/20 train/test split.
- Predefined subject assignments from `auxFiles/kdef_subject_split.pkl`.
- dlib 68-point facial landmarks.
- Separate `emotion_fuzzy` Conda environment.

## 1. Environment and dataset configuration

Resolve local data paths and load the team's predefined subject split.

In [5]:
from pathlib import Path
import os
import pickle

import numpy as np
import pandas as pd
import cv2
import dlib

# Locate the repository regardless of the notebook working directory.
cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "auxFiles/kdef_subject_split.pkl").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Repository root not found")

# Local paths; KDEF_ROOT can be overridden for other team members.
KDEF_ROOT = Path(
    os.getenv("KDEF_ROOT", str(ROOT.parent / "KDEF"))
).expanduser().resolve()

SPLIT_FILE = ROOT / "auxFiles/kdef_subject_split.pkl"
PREDICTOR_FILE = ROOT / "models/faceAndLandmarkModels/shape_predictor_68_face_landmarks.dat"
DATA_OUT = ROOT / "data/fuzzy"
OUT = ROOT / "outputs/fuzzy"

for path in (KDEF_ROOT, SPLIT_FILE, PREDICTOR_FILE):
    if not path.exists():
        raise FileNotFoundError(path)

for path in (DATA_OUT, OUT):
    path.mkdir(parents=True, exist_ok=True)

# Reuse the team's predefined subject-independent partition.
with SPLIT_FILE.open("rb") as f:
    split = pickle.load(f)

train_subjects = set(split["train_subjects"])
test_subjects = set(split["test_subjects"])

assert train_subjects.isdisjoint(test_subjects), "Subject leakage detected"
assert (len(train_subjects), len(test_subjects)) == (56, 14)

print(f"KDEF: {KDEF_ROOT}")
print(f"Split: {len(train_subjects)} train / {len(test_subjects)} test subjects")
print(f"NumPy: {np.__version__} | dlib: {dlib.__version__}")

KDEF: /Users/henriquedasilvacouto/workspace/TUe-Master_AI_and_ES/Masters-database/5ARE0_2026-GS1/Assignment_2/KDEF
Split: 56 train / 14 test subjects
NumPy: 1.24.4 | dlib: 19.24.2


## 2. KDEF metadata — Frontal images

Parse the original KDEF filenames into subject identity,
acquisition session, emotion, pose, and image path.

Both acquisition sessions belong to the same subject.
The filename corrections follow the team's existing KDEF loader.

In [6]:
EMOTIONS = {
    'AF': 'afraid', 'AN': 'angry', 'DI': 'disgust',
    'HA': 'happy', 'NE': 'neutral', 'SA': 'sad', 'SU': 'surprise'
}
OVERRIDES = {'AF31V': 'AF31SAHL', 'AM31H': 'AM31SUHR'}

rows, rejected = [], []
for path in sorted(KDEF_ROOT.rglob('*')):
    if not path.is_file() or path.suffix.lower() not in ('.jpg', '.jpeg'):
        continue
    stem = OVERRIDES.get(path.stem.upper(), path.stem.upper())
    if len(stem) < 7:
        rejected.append((str(path), 'short filename'))
        continue
    emotion_code = stem[4:6]
    pose_code = stem[6:]
    if emotion_code not in EMOTIONS or pose_code not in {'FL','HL','S','HR','FR'}:
        rejected.append((str(path), f'unrecognized emotion/pose: {stem}'))
        continue
    if pose_code != 'S':
        continue
    rows.append({
        'path': str(path),
        'image_name': path.name,
        'subject': stem[1:4],
        'acquisition': stem[0],
        'emotion': EMOTIONS[emotion_code],
        'pose': 'straight',
    })

meta = pd.DataFrame(rows)

# Verify the expected frontal-image dataset.
assert len(meta) == 980
assert meta["subject"].nunique() == 70
assert meta["emotion"].value_counts().eq(140).all()

print(f"Loaded {len(meta)} frontal images from {meta['subject'].nunique()} subjects")
print(meta["emotion"].value_counts().sort_index())

if rejected:
    print(f"Unrecognized filenames: {len(rejected)}")
    print(rejected[:5])

meta.head()

Loaded 980 frontal images from 70 subjects
afraid      140
angry       140
disgust     140
happy       140
neutral     140
sad         140
surprise    140
Name: emotion, dtype: int64


,path,image_name,subject,acquisition,emotion,pose
0,/Users/henriquedasilvacouto/workspace/TUe-Mast...,AF01AFS.JPG,F01,A,afraid,straight
1,/Users/henriquedasilvacouto/workspace/TUe-Mast...,AF01ANS.JPG,F01,A,angry,straight
2,/Users/henriquedasilvacouto/workspace/TUe-Mast...,AF01DIS.JPG,F01,A,disgust,straight
3,/Users/henriquedasilvacouto/workspace/TUe-Mast...,AF01HAS.JPG,F01,A,happy,straight
4,/Users/henriquedasilvacouto/workspace/TUe-Mast...,AF01NES.JPG,F01,A,neutral,straight
